In [1]:
import sys
# sys.path.append("/Users/bubble/Desktop/Project/Infrasound Sensor/Layout/Code/acousticsensor")
import math
import gdsfactory as gf
from mid_structure_square import create_mid_structure_square 

cell_temp = gf.Component()
# the whole chip
block = gf.Component()

In [2]:
# web mesh
mesh_web = gf.Component()
L = 1000
w = 2
g = 48

n_horizontal = 20
for i in range(20):
    h_horizontal = 300+i*50
    L_horizontal = 2*h_horizontal*math.tan(22.5*math.pi/180)
    s_horizontal = gf.Section(width=2, layer=(9, 0))
    X_horizontal = gf.CrossSection(sections=[s_horizontal])
    p_horizontal = gf.path.straight(L_horizontal)
    c_horizontal = gf.path.extrude(p_horizontal, X_horizontal)
    (mesh_web << c_horizontal).move((-L_horizontal/2, -h_horizontal))
    (mesh_web << c_horizontal).move((-L_horizontal/2, h_horizontal)).rotate(angle=45, center=(0, 0))

n_support = 5
for i in range(n_support):
    angle = 45/4*i-22.5
    L_support_1 = 1010/math.cos(angle*math.pi/180)
    L_support_2 = 950/math.cos(angle*math.pi/180)
    s_support = gf.Section(width=2, layer=(9, 0))
    X_support = gf.CrossSection(sections=[s_support])
    p_support = gf.path.straight(L_support_1)
    c_support_1 = gf.path.extrude(p_support, X_support)
    (mesh_web << c_support_1).rotate(-90).movey(-300/math.cos(angle*math.pi/180)).rotate(angle=angle, center=(0,0))
    p_support = gf.path.straight(L_support_2)
    c_support_2 = gf.path.extrude(p_support, X_support)
    (mesh_web << c_support_2).rotate(-90).movey(-300/math.cos(angle*math.pi/180)).rotate(angle=angle+45, center=(0,0))
# circle array
for i in range(4):
    angle = 360/4*i
    (block << mesh_web).rotate(angle=angle, center=(0, 0))
# length mark
T = gf.components.text(f"L={L} b={w} gap={g}", size=50, layer=(1, 0))
(block << T).move((-375, -1500))

# block.show()


Unnamed_1: ports [], vinsts=[] info=Info() kcl=KCLayout(name='DEFAULT', layout=<klayout.dbcore.Layout object at 0x10b2f07d0>, layer_enclosures=LayerEnclosureModel(root={}), cross_sections={}, enclosure=KCellEnclosure(enclosures=LayerEnclosureCollection(enclosures=[])), library=<klayout.dbcore.Library object at 0x10b2f06d0>, factories={'taper': <function taper at 0x1128da840>, 'bend_s_bezier': <function bend_s_bezier_factory.<locals>.bend_s_bezier at 0x10e0aff60>, 'bend_circular': <function bend_circular at 0x1128d9080>, 'bend_euler': <function bend_euler at 0x1128d9b20>, 'bend_s_euler': <function bend_s_euler_factory.<locals>.bend_s_euler at 0x110e15120>, 'straight': <function straight at 0x112947c40>, 'rotate': <function rotate at 0x11235a2a0>, 'mirror': <function mirror at 0x11235a3e0>, 'from_image': <function from_image at 0x112600680>, 'floorplan_with_block_letters': <function floorplan_with_block_letters at 0x112850400>, 'bend_circular_heater': <function bend_circular_heater at 0x

In [3]:
# note
"""
layer 9: structure to keep
layer 11: gold deposition area
layer 12: frontside etching area
"""

'\nlayer 9: structure to keep\nlayer 11: gold deposition area\nlayer 12: frontside etching area\n'

In [ ]:
# mid structure
mid_struct = create_mid_structure_square()
# mid_struct.show()
block << mid_struct

Unnamed_1: ports [], vinsts=[] info=Info() kcl=KCLayout(name='DEFAULT', layout=<klayout.dbcore.Layout object at 0x10b2f07d0>, layer_enclosures=LayerEnclosureModel(root={'4fb2b992': LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), '76156b2a': LayerEnclosure(layer_sections={}, main_layer=11/0, yaml_tag='!Enclosure')}), cross_sections={'4fb2b992_100000': SymmetricalCrossSection(width=100000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_100000'), '76156b2a_95000': SymmetricalCrossSection(width=95000, enclosure=LayerEnclosure(layer_sections={}, main_layer=11/0, yaml_tag='!Enclosure'), name='76156b2a_95000'), '4fb2b992_2000': SymmetricalCrossSection(width=2000, enclosure=LayerEnclosure(layer_sections={}, main_layer=9/0, yaml_tag='!Enclosure'), name='4fb2b992_2000')}, enclosure=KCellEnclosure(enclosures=LayerEnclosureCollection(enclosures=[])), library=<klayout.dbcore.Library object at 0x10b2f06d0>, factories={'tap

In [5]:
# frontside etching area
frame_size = 2600
frontside = gf.components.rectangle(size=(frame_size, frame_size), layer=(12, 0))
(block << frontside).move((-1300, -1300))

# backside etching area
backside_size = frame_size + 743.44
backside = gf.components.rectangle(size=(backside_size, backside_size), layer=(3, 0))
(block << backside).move((-backside_size/2, -backside_size/2))

# side frame etching area
# 7300um x 7300um gap between frame: 600
etch_frame = gf.components.rectangle(size=(6100, 375), layer=(3, 0))
for i in range(4):
    angle = 90 * i
    (block << etch_frame).move((-6100/2, -7300/2)).rotate(angle=angle, center=(0, 0))
# block.show()

In [ ]:
def create_simulation_structure():
    simu_structure = gf.boolean(A = block, B = block,  operation="not", layer1=(9, 0), layer2=(100, 0), layer=(1, 0))
    return simu_structure

In [6]:
# order
def add_order_text(order_number):
    order = gf.Component()
    for j in range(4):
        T = gf.components.text(f"WD{order_number}", size=20, layer=(1, 0))
        order_ref = order << T
        if j == 0:
            order_ref.move((-2000, -2000))
        elif j == 1:
            order_ref.move((-2000, 2000))
        elif j == 2:
            order_ref.move((2000, -2000))
        else:
            order_ref.move((2000, 2000))
    return order
    # order.show()
        

In [7]:
# repeat
fblock = gf.Component()
block_temp = gf.Component()
block_temp << block
for i in range(1):
    if i == 0:
        block_ref = fblock << block_temp
    elif i == 1:
        block_ref = fblock << block_temp
        block_ref.move((10000, 0))
    elif i == 2:
        block_ref = fblock << block_temp
        block_ref.move((10000, 10000))
    else:
        block_ref = fblock << block_temp
        block_ref.move((0, 10000))

In [8]:
# boolean operation
outside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(12, 0), layer2=(9, 0), layer=(1, 0))
cell_temp << outside

#  gold
gold = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(11, 0), layer2=(30, 0), layer=(2, 0))
cell_temp << gold
# backside etching
backside = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(3, 0), layer2=(30, 0), layer=(3, 0))
cell_temp << backside
# add marker/order
marker = gf.boolean(A = fblock, B = fblock,  operation="not", layer1=(1, 0), layer2=(30, 0), layer=(1, 0))
cell_temp << marker

def cell_web_design(order_number):
    cell_web_design = gf.Component()
    cell_temp_temp = gf.Component()
    order = add_order_text(order_number)
    cell_temp_temp << order
    cell_temp_temp << cell_temp
    cell_web_design << cell_temp_temp
    return cell_web_design
    # cell_ref.move((2500, -7500))
    # cell_web_design.show()
    # cell_web_design.write_gds("mesh.gds")
    # cell_web_design.plot()

if __name__ == "__main__":
    cell = cell_web_design(order_number=1)
    cell.show()